In [20]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.config import ScenarioConfig
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.fault_injection import SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.sweep_utils import (
    run_condition_sweep, summarize_outcomes,
    run_real_channel_trial, run_real_channel_reconciliation_trial, collect_key_pairs,
)

print("Setup complete")

Setup complete


In [21]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO = 'validation/scenarios/fabric_1km.yml'

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

alice_mac = alice.get_interface(network_name="net_alice_switch").get_mac()
bob_mac = bob.get_interface(network_name="net_switch_bob").get_mac()
sw_alice_mac = slice_obj.get_node("switch").get_interface(network_name="net_alice_switch").get_mac()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,900e9b70-47f3-4394-a831-e522acb47878
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-08-25 21:17:14 +0000
Lease Start (UTC),2026-08-11 21:17:14 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


In [22]:
alice_key, alice_indices = key_from_sifted_json(
    str(PROJECT_DIR / "results" / "fabric_alice_sifted_bits.json"), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(
    str(PROJECT_DIR / "results" / "fabric_bob_sifted_bits.json"), "bob_bits")

assert alice_indices == bob_indices, "alice and bob's matching indices don't match"

real_qber = alice_key.nr_bits_different(bob_key) / alice_key.get_nr_bits()
print(f"Loaded {alice_key.get_nr_bits()} bits, real_qber = {real_qber:.4f}")

Loaded 3876 bits, real_qber = 0.0114


In [ ]:
conditions = [
    ("baseline", {}),
    ("toeplitz_only", {"toeplitz_prob": 0.5}),
    ("hash_only", {"hash_prob": 0.5}),
    ("reconciliation_only", {"reconciliation_prob": 0.1}),
    ("combined", {"toeplitz_prob": 0.5, "hash_prob": 0.5, "reconciliation_prob": 0.1}),
]

all_dfs, summary_rows = {}, []
for label, kwargs in conditions:
    print(f"\nRunning {label}...")
    df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=10, **kwargs)
    all_dfs[label] = df_cond
    summary_rows.append(summarize_outcomes(df_cond, label))

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

full_df = pd.concat(all_dfs.values(), ignore_index=True)
full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_key_results.csv"), index=False)

In [ ]:
recon_probs = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
recon_dfs, recon_summary_rows = [], []

for prob in recon_probs:
    label = f"reconciliation_real_{prob}"
    df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=10, reconciliation_prob=prob)
    recon_dfs.append(df_cond)
    recon_summary_rows.append(summarize_outcomes(df_cond, label))

recon_summary_df = pd.DataFrame(recon_summary_rows)
print(recon_summary_df.to_string(index=False))

recon_full_df = pd.concat(recon_dfs, ignore_index=True)
recon_full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_key_reconciliation_doseresponse.csv"), index=False)

In [ ]:
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("hash_only", {"hash_prob": 0.5})]
all_rows = []

for label, kwargs in conditions:
    print(f"\n=== {label} on real channel ===")
    for run in range(10):
        seed = 42 + run
        result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, **kwargs)
        result["condition"] = label
        all_rows.append(result)
        print(f"  run {run}: keys_match={result.get('keys_match')}")

df = pd.DataFrame(all_rows)
df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_hash.csv"), index=False)

In [ ]:
existing_df = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_hash.csv"))
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("hash_only", {"hash_prob": 0.5})]
new_rows = []

for label, kwargs in conditions:
    for run in range(10, 30):
        seed = 42 + run
        result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, **kwargs)
        result["condition"] = label
        new_rows.append(result)

combined_df = pd.concat([existing_df, pd.DataFrame(new_rows)], ignore_index=True)
combined_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_hash.csv"), index=False)

In [ ]:
key_pairs_df = collect_key_pairs(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR, n_keys=5)
key_pairs_df.to_csv(str(PROJECT_DIR / "results" / "key_pairs_index.csv"), index=False)

In [33]:
#run if collecting new keys

alice.upload_file(str(PROJECT_DIR / "results" / "alice_sifted_bits_key0.json"),
                    "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(PROJECT_DIR / "results" / "alice_sifted_bits_key0.json"),
                  "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(PROJECT_DIR / "results" / "bob_sifted_bits_key0.json"),
                  "qfabric/results/bob_sifted_bits.json")

<SFTPAttributes: [ size=33925 uid=1000 gid=1000 mode=0o100644 atime=1786732656 mtime=1786733220 ]>

In [ ]:
recon_rows = []
for run in range(10):
    seed = 42 + run
    result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, real_qber, run, seed,
                                                       reconciliation_prob=0.03)
    recon_rows.append(result)
    print(f"run {run}: non_convergent={result.get('non_convergent')}, "
          f"faults_fired={result.get('faults_fired')}")

recon_df = pd.DataFrame(recon_rows)
print(recon_df.to_string(index=False))
recon_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_key0.csv"), index=False)

In [39]:
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("hash_only", {"hash_prob": 0.5})]

all_rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    # Point both nodes at this key
    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")

    a_key, a_idx = key_from_sifted_json(str(PROJECT_DIR / "results" / a_file), "alice_bits")
    b_key, b_idx = key_from_sifted_json(str(PROJECT_DIR / "results" / b_file), "bob_bits")
    assert a_idx == b_idx, f"key{key_idx}: indices don't match!"
    qber_i = a_key.nr_bits_different(b_key) / a_key.get_nr_bits()

    print(f"\n=== key{key_idx}: {a_key.get_nr_bits()} bits, QBER={qber_i:.4f} ===")

    for label, kwargs in conditions:
        print(f"  Running {label}...")
        for run in range(5):  # 5 trials per key per condition, given 5 keys x 2 conditions x 5 = 50 trials total
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, **kwargs)
            result["condition"] = label
            result["key_index"] = key_idx
            result["qber"] = qber_i
            all_rows.append(result)
            print(f"    run {run}: keys_match={result.get('keys_match')}, "
                  f"faults_fired={result.get('faults_fired')}")

df_multikey = pd.DataFrame(all_rows)
df_multikey.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_hash_multikey.csv"), index=False)
print(f"\nSaved {len(df_multikey)} rows.")


=== key0: 3810 bits, QBER=0.0110 ===
  Running toeplitz_only...
    run 0: keys_match=True, faults_fired={}
    run 1: keys_match=True, faults_fired={'toeplitz_matrix': 1}
    run 2: keys_match=False, faults_fired={'toeplitz_matrix': 1}
    run 3: keys_match=True, faults_fired={'toeplitz_matrix': 1}
    run 4: keys_match=False, faults_fired={'toeplitz_matrix': 1}
  Running hash_only...
    run 0: keys_match=True, faults_fired={}
    run 1: keys_match=True, faults_fired={}
    run 2: keys_match=False, faults_fired={'hash_output': 1}
    run 3: keys_match=True, faults_fired={}
    run 4: keys_match=False, faults_fired={'hash_output': 1}

=== key1: 3794 bits, QBER=0.0095 ===
  Running toeplitz_only...
    run 0: keys_match=True, faults_fired={}
    run 1: keys_match=False, faults_fired={'toeplitz_matrix': 1}
    run 2: keys_match=True, faults_fired={'toeplitz_matrix': 1}
    run 3: keys_match=True, faults_fired={}
    run 4: keys_match=True, faults_fired={}
  Running hash_only...
    run

In [40]:
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]
probs = [0.5, 0.3, 0.1, 0.05, 0.01]

all_rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")

    a_key, a_idx = key_from_sifted_json(str(PROJECT_DIR / "results" / a_file), "alice_bits")
    b_key, b_idx = key_from_sifted_json(str(PROJECT_DIR / "results" / b_file), "bob_bits")
    assert a_idx == b_idx
    qber_i = a_key.nr_bits_different(b_key) / a_key.get_nr_bits()
    print(f"\n=== key{key_idx}: QBER={qber_i:.4f} ===")

    for prob in probs:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}), ("hash", {"hash_prob": prob})]:
            print(f"  {fault_type} prob={prob}...")
            for run in range(3):  # 3 trials per key/prob/type, given the growing trial count
                seed = 42 + run
                result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)

df_doseresponse = pd.DataFrame(all_rows)
df_doseresponse.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_hash_doseresponse_multikey.csv"), index=False)
print(f"\nSaved {len(df_doseresponse)} rows.")


=== key0: QBER=0.0110 ===
  toeplitz prob=0.5...
  hash prob=0.5...
  toeplitz prob=0.3...
  hash prob=0.3...
  toeplitz prob=0.1...
  hash prob=0.1...
  toeplitz prob=0.05...
  hash prob=0.05...
  toeplitz prob=0.01...
  hash prob=0.01...

=== key1: QBER=0.0095 ===
  toeplitz prob=0.5...
  hash prob=0.5...
  toeplitz prob=0.3...
  hash prob=0.3...
  toeplitz prob=0.1...
  hash prob=0.1...
  toeplitz prob=0.05...
  hash prob=0.05...
  toeplitz prob=0.01...
  hash prob=0.01...

=== key2: QBER=0.0138 ===
  toeplitz prob=0.5...
  hash prob=0.5...
  toeplitz prob=0.3...
  hash prob=0.3...
  toeplitz prob=0.1...
  hash prob=0.1...
  toeplitz prob=0.05...
  hash prob=0.05...
  toeplitz prob=0.01...
  hash prob=0.01...

=== key3: QBER=0.0097 ===
  toeplitz prob=0.5...
  hash prob=0.5...
  toeplitz prob=0.3...
  hash prob=0.3...
  toeplitz prob=0.1...
  hash prob=0.1...
  toeplitz prob=0.05...
  hash prob=0.05...
  toeplitz prob=0.01...
  hash prob=0.01...

=== key4: QBER=0.0105 ===
  toeplit

In [42]:
"""
Real-channel reconciliation dose-response: fill out the sweep to match
the density of the Mock-based curve (6 probability points), appending
to whatever's already saved.
"""
import pandas as pd

existing_path = PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_key0.csv"
existing_recon = pd.read_csv(str(existing_path)) if existing_path.exists() else pd.DataFrame()

already_have = set(existing_recon["reconciliation_prob"].unique()) if len(existing_recon) else set()
print(f"Already have data for: {already_have}")

target_probs = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
probs_to_run = [p for p in target_probs if p not in already_have]
print(f"Running: {probs_to_run}")

new_rows = []
for prob in probs_to_run:
    print(f"\n=== reconciliation_prob={prob} ===")
    for run in range(10):
        seed = 42 + run
        result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, real_qber, run, seed,
                                                           reconciliation_prob=prob)
        new_rows.append(result)
        print(f"  run {run}: non_convergent={result.get('non_convergent')}, "
              f"faults_fired={result.get('faults_fired')}")

new_df = pd.DataFrame(new_rows)
combined = pd.concat([existing_recon, new_df], ignore_index=True)
combined.to_csv(str(existing_path), index=False)
print(f"\nSaved {len(combined)} total rows -> {existing_path}")

# Quick summary
summary = combined.groupby("reconciliation_prob")["non_convergent"].agg(
    ['mean', 'count']).rename(columns={'mean': 'non_convergent_rate'})
print(summary)

Already have data for: {np.float64(0.03)}
Running: [0.3, 0.1, 0.01, 0.003, 0.001]

=== reconciliation_prob=0.3 ===
  run 0: non_convergent=False, faults_fired={'reconciliation_state': 14}
  run 1: non_convergent=False, faults_fired={'reconciliation_state': 11}
  run 2: non_convergent=False, faults_fired={'reconciliation_state': 13}
  run 3: non_convergent=False, faults_fired={'reconciliation_state': 7}
  run 4: non_convergent=False, faults_fired={'reconciliation_state': 10}
  run 5: non_convergent=False, faults_fired={'reconciliation_state': 11}
  run 6: non_convergent=False, faults_fired={'reconciliation_state': 13}
  run 7: non_convergent=False, faults_fired={'reconciliation_state': 13}
  run 8: non_convergent=False, faults_fired={'reconciliation_state': 13}
  run 9: non_convergent=False, faults_fired={'reconciliation_state': 9}

=== reconciliation_prob=0.1 ===
  run 0: non_convergent=False, faults_fired={'reconciliation_state': 2}
  run 1: non_convergent=False, faults_fired={'reconc

In [43]:
df_check = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_key0.csv"))
print(df_check[df_check["reconciliation_prob"] == 0.3][["run", "total_corrections", "remaining_errors_after_reconciliation", "faults_fired"]])

    run  total_corrections  remaining_errors_after_reconciliation  \
10    0                 39                                     16   
11    1                 41                                     11   
12    2                 40                                     14   
13    3                 41                                      7   
14    4                 41                                     10   
15    5                 34                                     18   
16    6                 39                                     15   
17    7                 41                                     13   
18    8                 41                                     13   
19    9                 40                                     10   

                    faults_fired  
10  {'reconciliation_state': 14}  
11  {'reconciliation_state': 11}  
12  {'reconciliation_state': 13}  
13   {'reconciliation_state': 7}  
14  {'reconciliation_state': 10}  
15  {'reconciliation_state': 1

In [48]:
df_check = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_key0.csv"))
print(df_check["elapsed_seconds"].describe())

count    60.000000
mean     53.901563
std       1.247947
min      51.646508
25%      53.074661
50%      53.979470
75%      54.566842
max      58.600310
Name: elapsed_seconds, dtype: float64
